# 把遊樂場做好的 Agent 帶回程式碼這份教材的主題是讀懂遊樂場匯出的程式碼，確認哪些設定要保存，並把可信任的匯出內容帶回 SDK 使用。

## 在 Colab 準備環境先準備 SDK 執行環境。

In [ ]:
from pathlib import Pathimport os, sys, subprocessif not Path('agentic_sdk').exists():    if not Path('Agentic-SDK').exists():        subprocess.run(['git', 'clone', 'https://github.com/R300-AI/Agentic-SDK.git'], check=True)    os.chdir('Agentic-SDK')subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)print('Agentic SDK ready')

## 先放一段遊樂場匯出的程式碼遊樂場匯出的內容，本質上是把 Builder 裡的設定轉成 SDK 物件。這裡不另外教每個基礎模組，只先把匯出內容當成文字來看。

In [ ]:
exported_source = '''from agentic_sdk import Workflowfrom agentic_sdk.modules import DirectAnswerAction, KeywordRetrieve, PassThroughPerceiveworkflow = Workflow(    workflow_name='AI Hub 入門助手',    description='回答 AI Hub 上架流程相關問題。',    perceive=PassThroughPerceive(),    retrieve=KeywordRetrieve(items=[        {'keywords': ['上架', '保存'], 'content': '完成設定後，請在 Runner 按下保存。'},    ]),    action=DirectAnswerAction(),)'''print(exported_source)

## 先找出需要帶回專案的設定不要急著執行。先看工作流程名稱、說明、查資料設定和回答方式是否符合你在遊樂場裡做的設定。

In [ ]:
for line in exported_source.splitlines():    if 'workflow_name' in line or 'description' in line or 'KeywordRetrieve' in line or 'DirectAnswerAction' in line:        print(line)

## 確認來源可信任後再執行`exec` 只應該用在你信任來源的匯出程式碼。正式產品可以改成白名單解析或受控載入。

In [ ]:
namespace = {}exec(exported_source, namespace)workflow = namespace['workflow']workflow

## 用匯出的工作流程試跑現在 `workflow` 已經從匯出程式碼建立完成，可以像一般 SDK 物件一樣呼叫。

In [ ]:
result = workflow.run('完成設定後要做什麼？')print(result.final_message)

## 最後確認保存資料不要漏只保存程式碼不一定夠。如果 Agent 使用參考文件與查詢索引，也要保存 bundle，重新打開時才會完整。

In [ ]:
save_checklist = [    '工作流程程式碼',    '工作流程名稱與摘要',    '參考文件',    '查詢索引或可重建索引的來源資料',    'AI Hub agent_id 與權限狀態',]for item in save_checklist:    print('-', item)